# Phase5: もっと良い方法は？ — 自動拡張・画像を混ぜる・転移学習（最終回）

> ⏱ 所要時間: 説明を読む 12分 ＋ 実行 約15分（`QUICK = True`）。転移学習の初回は、学習済みAIの重みファイル（約45MB）を自動でダウンロードします。

## このNotebookのゴール
1. 拡張の選び方を**おまかせ**にできる方法（**自動拡張**）を知る
2. **画像を混ぜる**強い拡張（Mixup / CutMix）が、どんなときに**向かない**かを知る
3. **転移学習**が、少ない写真のときに**拡張よりはるかに効く**ことを体験する
4. 演習全体の**「実践的な指針」**（自分の課題で何から試すか）をまとめる

## 前のNotebookとのつながり
Notebook 04 で、アリ/ハチには**切り抜き・反転**が効くことが分かりました。ここまでは**人間が拡張を選んで**いましたが、
今回は**自動で選ぶ方法**と、**まったく別のアプローチ（転移学習）**を試します。

## キーワード
| 用語 | 意味 |
|---|---|
| **自動拡張（RandAugment / TrivialAugment）** | 「どの拡張をどの強さで」をランダムに自動で決める手法。人が調整しなくてよい |
| **Mixup** | 2枚の写真を**重ねて混ぜ**、ラベルも混ぜる拡張 |
| **CutMix** | 写真の一部を**別の写真に貼り替え**、ラベルは面積の割合で混ぜる拡張 |
| **ソフトラベル** | 「アリ80%・ハチ20%」のような、混ざったラベル |
| **転移学習** | 大量の写真（ImageNet、約100万枚）で学習済みのAIを借りて、自分の課題用に**少しだけ**調整する方法 |
| **ResNet18** | 有名な画像AIの1つ。今回の転移学習で借りる |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

## 設定

In [ ]:
QUICK = True                          # True: 各条件1回 / False: 3回の平均
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)
from torchvision.transforms import v2
from torchvision import datasets
import torch, random

## 1. 自動拡張の見た目

これまでは「切り抜き」「反転」など、**人間が選んで**拡張を決めてきました。**自動拡張**は、選ぶ作業をおまかせにします。

| 手法 | しくみ |
|---|---|
| **TrivialAugment** | 毎回、**操作を1つ**ランダムに選び、**強さもランダム**に決める。調整するパラメータが**ない** |
| **RandAugment** | 毎回、操作を**N個**（今回は2個）選び、強さMで適用する。調整するのは N と M だけ |

次のセルは、**同じ1枚の写真**に4回ずつ適用して、毎回違う結果になることを確認します（上段: TrivialAugment、下段: RandAugment）。

In [ ]:
from PIL import Image
random.seed(1); torch.manual_seed(1)
img = v2.Resize((224, 224))(Image.open(sorted((DATA / "train/ants").glob("*.jpg"))[3]).convert("RGB"))

show_images([v2.TrivialAugmentWide()(img) for _ in range(4)], [f"Trivial {i+1}" for i in range(4)], cols=4, size=2.6)
show_images([v2.RandAugment(num_ops=2, magnitude=9)(img) for _ in range(4)], [f"RandAug {i+1}" for i in range(4)], cols=4, size=2.6);

**見るポイント**: 色の変化、回転、明るさの変化など、**いろいろな加工が自動で**かかっています。人が選ばなくても多様な拡張ができます。

## 2. 画像を混ぜる拡張（Mixup / CutMix）の見た目

**Mixup** と **CutMix** は、これまでとは違う発想の拡張です。**2枚の写真を混ぜて、1枚の新しい写真**を作ります。

- **Mixup**: 2枚をピクセルごとに**重ね合わせる**（半透明に重ねたイメージ）
- **CutMix**: 1枚の一部を**四角く切り取って、別の写真に貼る**

そして、**ラベルも混ぜます**。たとえば「アリの写真の80%＋ハチの写真の20%」なら、ラベルは「アリ0.8・ハチ0.2」（**ソフトラベル**）になります。

次のセルで、**アリとハチを交互に8枚**用意し、Mixup と CutMix で混ぜた写真を表示します。タイトルの `ラベル=[0.8, 0.2]` は **[アリ, ハチ]の割合**です。

In [ ]:
ds = datasets.ImageFolder(DATA / "train", T.make_transform("cropflip"))
# アリ（0〜3番目）とハチ（130〜133番目あたり）を交互に8枚そろえる
idx = [0, 130, 1, 131, 2, 132, 3, 133]
x = torch.stack([ds[i][0] for i in idx]); y = torch.tensor([ds[i][1] for i in idx])
print("クラス（0番目, 1番目）:", ds.classes)

torch.manual_seed(0)
for name, mixer in [("Mixup", v2.MixUp(num_classes=2)), ("CutMix", v2.CutMix(num_classes=2))]:
    xm, ym = mixer(x, y)    # 混ぜた画像 xm と、混ぜたラベル ym
    show_images(list(xm[:4]), [f"{name} ラベル={[round(v,2) for v in ym[i].tolist()]}" for i in range(4)], cols=4, size=2.8)

**見るポイント**
- **Mixup**: 2枚が重なって、**どちらの写真か見分けにくい**。人間には難しい問題。
- **CutMix**: 大きな写真に、別の写真の一部が貼られている。**貼られた面積に応じて**ラベルの割合が決まる。
- どちらも**人間にとっては不自然な写真**ですが、AIに「**はっきりしない例でも、自信を持ちすぎない**」ことを教える効果があります。

## 3. ゼロから学習: 拡張を比べる（150エポック）

ここまでの結果を踏まえ、**基準を「切り抜き＋反転」**（`cropflip`）にして、自動拡張と Mixup / CutMix を比べます。
「ゼロから学習」とは、**何も知らないAI**を最初から学習させることです（あとで出てくる「転移学習」と区別するための言い方）。

| 条件 | 内容 |
|---|---|
| `cropflip` | 切り抜き＋反転（基準） |
| `trivial` | TrivialAugment |
| `randaug` | RandAugment |
| `cropflip+mixup` | 切り抜き＋反転 **に Mixup を加える** |
| `cropflip+cutmix` | 切り抜き＋反転 **に CutMix を加える** |

次のセルは5条件を学習します（`QUICK = True` で約7分）。

In [ ]:
configs = [("cropflip", "none"), ("trivial", "none"), ("randaug", "none"), ("cropflip", "mixup"), ("cropflip", "cutmix")]
scratch = []
for aug, mix in configs:
    print(f"== aug={aug} mix={mix} ==")
    d, _ = run_seeds(SEEDS, aug=aug, mix=mix, epochs=EPOCHS)
    scratch.append(d)
scratch = pd.concat(scratch)
mean_table(scratch)

### 結果の読み方（MANUAL.mdの記載値・3シード平均、test精度）

| 条件 | test |
|---|---|
| 拡張なし（参考） | 0.701 |
| 切り抜き＋反転 | 0.801 |
| TrivialAugment | 0.792 |
| **RandAugment** | **0.844** |
| 切り抜き＋反転＋Mixup | 0.753 |
| 切り抜き＋反転＋CutMix | 0.714 |

### 何が分かるか
1. **自動拡張（RandAugment）は、人が選んだ組み合わせと同等以上**（0.844）。**どの拡張を選べばいいか迷ったら、RandAugment を試す**のが近道。
   （ただし標準偏差が0.04〜0.06あり、切り抜き単独 0.835 との差は誤差の範囲）
2. **Mixup・CutMix は、逆効果**（0.753、0.714）。
   理由: AIが小さく、写真も少ないため、**混ぜた難しい写真を学習しきれない**（Mixup では train 精度が 0.69 まで下がり、**未学習**気味）。
   これらは**大きなAIを長く学習する場面**で効く強い手法で、**今回の規模には強すぎた**。

> 💡 **「新しい手法＝良い手法」ではない。** 手法には**向き不向き**があり、**自分の課題の規模で試して確かめる**ことが大切です。

## 4. 転移学習（学習済みAIを借りる）

ここまでは、**何も知らないAI**を、243枚の写真だけで学習させてきました。
**転移学習**は、まったく別の発想です。

> 🍎 たとえ話: 英語がまったく分からない人に、**フランス語**を教えるより、**すでに英語ができる人**に教えるほうが速い。英語を学んだ経験（単語の作り方・文の構造）が**役に立つ**から。

**ImageNet**（約100万枚・1000種類の写真）で**学習済みのAI（ResNet18）**は、すでに「線・角・模様・形」の見方を**知っています**。
これを借りて、アリ・ハチ用に**少しだけ調整（微調整）**します。

- 学習率は**小さめ**（`lr=1e-4`）: 借りた知識を壊さないよう、**そっと**調整する。
- 学習は**15エポック**で十分（もう下地があるので速い）。
- 初回は学習済みの重み（約45MB）を自動ダウンロードします。

条件: 拡張なし／切り抜き＋反転／TrivialAugment／切り抜き＋反転＋CutMix の4つを比べます。

In [ ]:
tl_configs = [("none", "none"), ("cropflip", "none"), ("trivial", "none"), ("cropflip", "cutmix")]
transfer = []
for aug, mix in tl_configs:
    print(f"== 転移学習(ResNet18) aug={aug} mix={mix} ==")
    d, _ = run_seeds(SEEDS, aug=aug, mix=mix, epochs=ep(15), model_name="resnet18", lr=1e-4)
    transfer.append(d)
transfer = pd.concat(transfer)
mean_table(transfer)

### 極少量データ（クラスあたり10枚）での転移学習
Notebook 04 では、**10枚**だと拡張の効果が出ませんでした。**転移学習なら、10枚でも戦えるか**を見ます。（アリ10枚＋ハチ10枚＝合計20枚で学習）

In [ ]:
tiny = []
for aug in ["none", "cropflip"]:
    print(f"== 転移学習 クラスあたり10枚 aug={aug} ==")
    d, _ = run_seeds(SEEDS, aug=aug, epochs=ep(15), model_name="resnet18", lr=1e-4, n_per_class=10)
    tiny.append(d)
tiny = pd.concat(tiny)
mean_table(tiny)

最後に、**「ゼロから学習」と「転移学習」の成績を並べて**比べます（拡張なし／拡張ありの最良に近いもの）。

In [ ]:
summary = pd.DataFrame({
    "ゼロから学習(拡張なし)":      [0.701],
    "ゼロから学習(RandAugment)":  [scratch[scratch.aug == "randaug"].test_acc.mean()],
    "転移学習(拡張なし)":          [transfer[transfer.aug == "none"].test_acc.mean()],
    "転移学習(切り抜き+反転)":      [transfer[transfer.aug == "cropflip"].test_acc.mean()],
}).T.rename(columns={0: "test精度"})
summary.round(3)   # 「ゼロから学習(拡張なし)」の0.701は MANUAL.md の記載値（Notebook 03）

### 結果の読み方（MANUAL.mdの記載値・3シード平均、test精度）

| データ | 条件 | test |
|---|---|---|
| 全243枚 | 拡張なし | **0.892** |
| 全243枚 | 切り抜き＋反転 | **0.939** |
| 全243枚 | TrivialAugment ／ +CutMix | 0.905 ／ 0.913 |
| 10枚/クラス | 拡張なし → 切り抜き＋反転 | 0.783 → 0.805 |

### 何が分かるか
1. **最大の改善要因は、転移学習。** 拡張なしでも **0.701 → 0.892**（+19ポイント）。10枚/クラスでも、ゼロから学習の **0.64 → 0.78**。
2. **拡張は、その上の「上積み」**（0.892 → 0.939、+4.7ポイント）。拡張だけでは届かない水準に、転移学習が連れて行ってくれる。
3. 転移学習では、**train 精度が1.000でも test が高い**。**「train が高い＝過学習＝悪い」とは限らない。** 大事なのは val / test の成績。
4. 転移学習では、TrivialAugment や CutMix が**切り抜き＋反転より低い**。「強い拡張」が常に良いわけではない。

## 🧪 試してみよう

### 演習1: RandAugment の強さを変える
**やること**: `src/train.py` を開き、`AUG_SETS` の `"randaug"` の **`magnitude=9`** を **`magnitude=3`** に書き換えて保存します。
**カーネルを再起動**し、このNotebookの**3. ゼロから学習**のセルだけ実行し直します。
- `magnitude` は拡張の**強さ**（0〜30程度）。`3` は弱い設定、`9` は標準、`15` は強い設定。
- **見るポイント**: 弱いと拡張の効果が薄い？ 強いと不自然な写真が増えて悪化する？
- ⚠ 終わったら `magnitude=9` に**戻してください**。

### 演習2: 転移学習の学習率を大きくする
**やること**: 「4. 転移学習」のセルの **`lr=1e-4`** を **`lr=1e-2`**（100倍）に書き換えて実行します。
- **見るポイント**: 精度が**下がります**。なぜ？ → 学習率が大きいと、**せっかく借りた知識を壊してしまう**から。
- **分かること**: 転移学習では、**学習率を小さく**して、知識をそっと調整する。

### 演習3: 自分の課題に当てはめて考える（コード不要）
あなたが「**自分の写真データ**（たとえば地衣類の写真）を分類したい。写真は200枚」という状況だとします。
下の「実践的な指針」を使って、**最初に何を試すか、順番に書き出してみましょう**。

---

## ✅ まとめ: 実践的な指針（この演習の結論）

少ない写真で画像分類をするときは、次の順に考えます。

| 順番 | やること | 理由（この演習で得た根拠） |
|---|---|---|
| **1** | **まず転移学習**（学習済みAIを借りる） | 拡張なしでも +19ポイント。拡張より、はるかに効く |
| **2** | 拡張は**切り抜き＋水平反転**から | ゼロから学習で +13ポイント。実装も簡単で、副作用が少ない |
| **3** | 迷ったら **RandAugment / TrivialAugment** | 調整なしで、手作りの組み合わせに匹敵した |
| **4** | 拡張は**1種類ずつ**試して効果を確かめる | 色調のように、無駄な拡張に気付ける |
| **5** | 拡張ありは学習が遅いので、**十分な回数**で比べる | 60エポックでは効果が見えず、150で出た |
| **6** | **Mixup / CutMix は小さなAIでは慎重に** | 未学習になり、逆効果だった |
| **7** | 拡張は**データの代わりにならない**。極端に少ないなら、**写真を増やす** | 10〜30枚/クラスでは、test が上がらなかった |
| **8** | 比較は**複数シードの平均**で。0.03程度の差は過信しない | val/test が少ないと、結果がぶれる |
| **9** | **分割は拡張の前、拡張は train だけ、test は最後に1回だけ** | データリークと、testでの調整（カンニング）を避ける |

詳しい数値と考察は、プロジェクトの **`MANUAL.md`** にまとまっています。

---

## 🎉 演習完了！

お疲れさまでした。**少ない写真でも、データ拡張と転移学習で、AIの精度を上げられること**を、実験で確かめました。
次は、**自分の課題の写真**（自分で撮影した写真など）で、この手順を試してみましょう。（写真に位置情報や人物が写っていないか、確認してから使ってください）